# Phase 9 — Reflection & Hallucination Guardrails

> **Gemini-only project.** Every cell here uses Google Gemini (chat + embeddings) via `langchain-google-genai`. The only key the core needs is `GOOGLE_API_KEY`. This notebook is a **scaffold** — run it top-to-bottom *after* Claude Code finishes this phase. If a cell references a module that doesn't exist yet, that phase hasn't been built.

**Goal:** catch an invented number and trigger a revision loop. LLM-judge guards run on Gemini.


In [ ]:
# --- Setup: load .env and make the project importable ---
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # repo root
from dotenv import load_dotenv
load_dotenv('../.env', override=True)  # override=True: beat VS Code's stale cached env vars

# This project is Gemini-only. Confirm the one required key is present.
assert os.getenv('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY in .env (see 00_prerequisites_and_setup.md)'
print('GOOGLE_API_KEY loaded:', bool(os.getenv('GOOGLE_API_KEY')))
print('SEC_USER_AGENT   :', os.getenv('SEC_USER_AGENT', '(set before Phase 5)'))

## 1. NumericConsistencyGuard catches a fabricated number (cheap, no LLM)

In [ ]:
from app.guardrails.hallucination_detector import NumericConsistencyGuard
state = {'final_answer': 'Your RSI is 72.', 'tool_results': {'rsi': 68.4}}
res = NumericConsistencyGuard().check(state)
print('passed:', res.passed, '| action:', res.action, '| reason:', res.reason)

## 2. Reflection loop revises the answer (Producer/Critic)

In [ ]:
from app.graph.builder import GraphBuilder
graph = GraphBuilder().with_all().build()
# Temporarily patch the synthesizer to emit a fake number, then observe the guard force a revise.
# (See app/graph/synthesizer.py; the phase prompt walks through this patch/unpatch.)
out = graph.invoke({'messages': [('user', 'What is my NVDA RSI?')], 'client_id': 'CLT-005',
                    'session_id': 'nb-9'}, {'configurable': {'thread_id': 'CLT-005-nb9'}})
print(out['messages'][-1].content)

## 3. Input guardrails (PII / injection / scope)

In [ ]:
from app.guardrails.input_guardrails import PIIGuard, ScopeGuard
print(ScopeGuard().check({'query': 'write me a poem'}))   # off-topic -> block
print(PIIGuard().check({'query': 'my SSN is 123-45-6789'}))  # redact

## ✅ Acceptance check

In [ ]:
assert res.action == 'revise'
print('Phase 9 acceptance: PASS (numeric hallucination caught)')